In [1]:
#run spark.stop before running this notebook. some of the configs won't take if an active spark sessions is grabbed or running. 
#spark.stop()

In [2]:
import psutil
import time
import threading
import datetime
import json

In [3]:
import pyspark
import os
import pandas as pd
from torchvision.io import read_image
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import datasets, models
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import pyarrow


from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import FloatType

import math
from statistics import NormalDist

In [4]:
class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = torch.nn.Sequential(
            #Input = 3 x 32 x 32, Output = 32 x 32 x 32 
            #new input 3x600x375 output=
            torch.nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 32 x 32 x 32, Output = 32 x 16 x 16 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 32 x 16 x 16, Output = 64 x 16 x 16 
            torch.nn.Conv2d(in_channels = 6, out_channels = 12, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 16 x 16, Output = 64 x 8 x 8 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8 
            torch.nn.Conv2d(in_channels = 12, out_channels = 24, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8 
            torch.nn.Conv2d(in_channels = 24, out_channels = 48, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8 
            torch.nn.Conv2d(in_channels = 48, out_channels = 96, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4 
            torch.nn.MaxPool2d(kernel_size=2),

            torch.nn.Flatten(),

            torch.nn.Linear(96*11*18, 128),
            torch.nn.ReLU(),
            torch.nn.BatchNorm1d(128),




            #torch.nn.Linear(128, 128),
            #torch.nn.ReLU(),
            #torch.nn.BatchNorm1d(128),



            torch.nn.Linear(128, 2),
            torch.nn.ReLU(),
            #torch.nn.BatchNorm1d(2),
            #torch.nn.Dropout(0.2),


            torch.nn.Softmax(dim=1)

        )

    def forward(self, x):

            

        return self.model(x)

In [5]:
#tuning knob 1. put a very between 1 and 10, depending on sys cores. this also reparitions the df for sql queries
threads = 4

In [8]:
spark = pyspark.sql.SparkSession.builder.master(f"local[{threads}]").appName("silver").config("spark.executor.memory", "24g").config("spark.driver.memory", "24g").getOrCreate()

print(spark)

In [10]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNN().to(device)
model.load_state_dict(torch.load('../mymodel', map_location=torch.device(device)))
model.eval() # Set the model to evaluation mode

bc_model = spark.sparkContext.broadcast(model)

In [12]:
res18_model = models.resnet18(pretrained=True).to(device)
res18_model.fc = nn.Linear(res18_model.fc.in_features, 2 )
res18_model.load_state_dict(torch.load('../resnet18_model.pth', map_location=torch.device(device)))
res18_model.eval() # Set the model to evaluation mode

bc_res18_model = spark.sparkContext.broadcast(res18_model)

In [13]:
print(device)

cpu


NORMAL UDF (doesnt work in spark sql)

In [14]:
def predict_tensor(tensor_data):
    tensor = torch.Tensor(tensor_data)
    
    #channel, h, w = tensor.shape
  
    with torch.no_grad():  # Disable gradient calculation for inference
        prediction = model(tensor.unsqueeze(0).to(device))
        _, y_pred = torch.max(prediction.data, 1)
        
    return float(y_pred)
    

    
    

predict_udf = udf(predict_tensor, FloatType())

In [15]:
spark.udf.register("predict_udf", predict_udf)

In [16]:
def predict_res18_tensor(tensor_data):
    tensor = torch.Tensor(tensor_data)
    
    #channel, h, w = tensor.shape
  
    with torch.no_grad():  # Disable gradient calculation for inference
        prediction = res18_model(tensor.unsqueeze(0).to(device))
        _, y_pred = torch.max(prediction.data, 1)
        
    return float(y_pred)
    

    
    

predict_udf = udf(predict_tensor, FloatType())

In [17]:
spark.udf.register("predict_res18_udf", predict_udf)

VECTORIZED UDF (WORKS IN SPARK SQL)

In [18]:
@pandas_udf(returnType=FloatType())
def v_predict_tensor(tensor_data):
    

    
    input_tensor = torch.Tensor(tensor_data.tolist())

  

    
    with torch.no_grad():  # Disable gradient calculation for inference
    
        predictions = model(input_tensor.to(device))
        _, y_pred = torch.max(predictions.data, 1)
    
    #return y_pred
    return pd.Series(y_pred.squeeze().tolist())


In [19]:
spark.udf.register("v_predict_udf", v_predict_tensor)

In [20]:
@pandas_udf(returnType=FloatType())
def v_predict_res18_tensor(tensor_data):
    

    
    input_tensor = torch.Tensor(tensor_data.tolist())

  

    
    with torch.no_grad():  # Disable gradient calculation for inference
    
        predictions = model(input_tensor.to(device))
        _, y_pred = torch.max(predictions.data, 1)
    
    #return y_pred
    return pd.Series(y_pred.squeeze().tolist())

In [21]:
spark.udf.register("v_predict_res18_udf", v_predict_res18_tensor)

In [22]:
#tuning knob 2. put a .limit(20) at the end of the data set loads to reduce memory footprint for dev stuff 
#all_images_labeled = spark.read.parquet("./silver/all_images_labeled.parquet")
labeled_training_set = spark.read.parquet("./silver/labeled_training_set.parquet")
labeled_training_set.cache()
labeled_training_set.count()

998

In [23]:
labeled_training_set.show()

+-------------------+--------------------+----------------+-------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+
|               name|          img_tensor|prelabeled_fraud|                 id|    person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|
+-------------------+--------------------+----------------+-------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+
|generated_fake_2_49|[[[0.98039216, 0....|               1|generated_fake_2_49| ABIGAIL MILLER|38 Silver Creek R...| 12/3/2006|     F|     NULL|ERROR|11/13/2021| 11/13/2026|      BRO|      NULL|   FALSE|     FALSE|     D69254528|      

In [24]:
labeled_training_set.select('name', 'id').show(n=1000, truncate=False)

+-------------------------------------------+-------------------------------------------+
|name                                       |id                                         |
+-------------------------------------------+-------------------------------------------+
|generated_fake_2_49                        |generated_fake_2_49                        |
|generated_fake_2_52                        |generated_fake_2_52                        |
|generated_fake_2_55                        |generated_fake_2_55                        |
|generated_fake_2_86                        |generated_fake_2_86                        |
|generated_fake_2_79                        |generated_fake_2_79                        |
|generated_fake_2_45                        |generated_fake_2_45                        |
|generated_fake_2_47                        |generated_fake_2_47                        |
|generated_fake_2_69                        |generated_fake_2_69                        |
|generated

examples of dataframe api udf calls. Faster than sql, but will need to do more dataframe manipulation thats not as easy as spark.sql

In [25]:
test = labeled_training_set.withColumn("mylabel", predict_udf(labeled_training_set['img_tensor']))
test.show()

+-------------------+--------------------+----------------+-------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+-------+
|               name|          img_tensor|prelabeled_fraud|                 id|    person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|mylabel|
+-------------------+--------------------+----------------+-------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+-------+
|generated_fake_2_49|[[[0.98039216, 0....|               1|generated_fake_2_49| ABIGAIL MILLER|38 Silver Creek R...| 12/3/2006|     F|     NULL|ERROR|11/13/2021| 11/13/2026|      BRO|      NULL|   FALSE|     FAL

In [26]:
test2 = labeled_training_set.withColumn("mylabel", v_predict_tensor(labeled_training_set['img_tensor']))
test2.show()

/tmp/ipykernel_226088/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)


+-------------------+--------------------+----------------+-------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+-------+
|               name|          img_tensor|prelabeled_fraud|                 id|    person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|mylabel|
+-------------------+--------------------+----------------+-------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+-------+
|generated_fake_2_49|[[[0.98039216, 0....|               1|generated_fake_2_49| ABIGAIL MILLER|38 Silver Creek R...| 12/3/2006|     F|     NULL|ERROR|11/13/2021| 11/13/2026|      BRO|      NULL|   FALSE|     FAL

In [27]:
#last tuning know, change reparition based on your system specs and the size of the data being read in. spark will usually do around 20 by default. may or may not work well for you. higher will protect against OOM errors. Lower will make your processing time alot higher
labeled_training_set = labeled_training_set.repartition(threads*10)
labeled_training_set.cache()
labeled_training_set.count()

998

In [28]:
#labeled_training_set.rdd.getNumPartitions()

examples of using udf in sql queries

In [29]:
spark.sql("select gender, predict_udf(img_tensor) as my_predictions from {labeled_training_set}", labeled_training_set=labeled_training_set).show()

+------+--------------+
|gender|my_predictions|
+------+--------------+
|     F|           0.0|
|     F|           0.0|
|     M|           1.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           0.0|
+------+--------------+
only showing top 20 rows


In [30]:
spark.sql("select gender, v_predict_udf(img_tensor) as my_predictions from {labeled_training_set}", labeled_training_set=labeled_training_set).show()

/tmp/ipykernel_226088/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)


+------+--------------+
|gender|my_predictions|
+------+--------------+
|     F|           0.0|
|     F|           0.0|
|     M|           1.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           0.0|
+------+--------------+
only showing top 20 rows


In [31]:
spark.sql("with cte as (select gender, predict_udf(img_tensor) as my_predictions from {labeled_training_set}) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions", labeled_training_set=labeled_training_set).show()

+------+--------------+-----------------+
|gender|my_predictions|count_predictions|
+------+--------------+-----------------+
|     F|           1.0|              104|
|     F|           0.0|              442|
|     M|           1.0|               92|
|     M|           0.0|              357|
| ERROR|           1.0|                3|
+------+--------------+-----------------+



In [32]:
spark.sql("with cte as (select gender, v_predict_udf(img_tensor) as my_predictions from {labeled_training_set}) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions", labeled_training_set=labeled_training_set).show()


/tmp/ipykernel_226088/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
/tmp/ipykernel_226088/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
/tmp/ipykernel_226088/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
/tmp/ipykernel_226088/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.

+------+--------------+-----------------+
|gender|my_predictions|count_predictions|
+------+--------------+-----------------+
|     F|           1.0|              104|
|     F|           0.0|              442|
|     M|           1.0|               92|
|     M|           0.0|              357|
| ERROR|           1.0|                3|
+------+--------------+-----------------+



In [33]:
#using res18
spark.sql("select gender, v_predict_res18_udf(img_tensor) as my_predictions from {labeled_training_set}", labeled_training_set=labeled_training_set).show()

+------+--------------+
|gender|my_predictions|
+------+--------------+
|     F|           0.0|
|     F|           0.0|
|     M|           1.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           0.0|
+------+--------------+
only showing top 20 rows


In [34]:
spark.sql("with cte as (select gender, v_predict_res18_udf(img_tensor) as my_predictions from {labeled_training_set}) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions", labeled_training_set=labeled_training_set).show()

/tmp/ipykernel_226088/2464696233.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)


+------+--------------+-----------------+
|gender|my_predictions|count_predictions|
+------+--------------+-----------------+
|     F|           1.0|              104|
|     F|           0.0|              442|
|     M|           1.0|               92|
|     M|           0.0|              357|
| ERROR|           1.0|                3|
+------+--------------+-----------------+



In [35]:
def pdf_udf(gender, weight_lbs):
    
    #def normpdf(x, mean, sd):
    #    var = float(sd)**2
    #    denom = (2*math.pi*var)**.5
    #    num = math.exp(-(float(x)-float(mean))**2/(2*var))
    #    return num/denom
    
    #randomly taken from google, do not claim this to be sci accurate on stats
    male_mean = 200
    male_dev = 40
    female_mean = 170
    female_dev = 51

    gender = gender.upper()
    
    if gender == 'M':
        a = NormalDist(male_mean, male_dev).cdf(int(weight_lbs)+5)
        b = NormalDist(male_mean, male_dev).cdf(int(weight_lbs)-5)
        probability_density = abs(a-b)
        
    else: 
        a = NormalDist(female_mean, female_dev).cdf(int(weight_lbs)+5)
        b = NormalDist(female_mean, female_dev).cdf(int(weight_lbs)-5)
        probability_density = abs(a-b)
    
    return round(probability_density, 3)

In [36]:
spark.udf.register("pdf_udf_sql", pdf_udf)

<function __main__.pdf_udf(gender, weight_lbs)>

In [37]:
spark.sql('select gender, weight_lbs, pdf_udf_sql(gender, weight_lbs) from {labeled_training_set}', labeled_training_set=labeled_training_set).show()

+------+----------+-------------------------------+
|gender|weight_lbs|pdf_udf_sql(gender, weight_lbs)|
+------+----------+-------------------------------+
|     M|       124|                          0.017|
|     F|       128|                          0.056|
|     F|       175|                          0.078|
|     M|       103|                          0.005|
|     M|       138|                           0.03|
|     M|       162|                          0.063|
|     M|       186|                          0.094|
|     M|       142|                          0.035|
|     F|       140|                          0.066|
|     F|       116|                          0.045|
|     M|       234|                          0.069|
|     M|       103|                          0.005|
|     M|       214|                          0.094|
|     F|       186|                          0.074|
|     M|       113|                          0.009|
|     F|       157|                          0.076|
|     F|    

In [38]:
spark.sql("with cte1 as (select gender, weight_lbs, img_tensor, pdf_udf_sql(gender, weight_lbs) as w_prob from {labeled_training_set}), cte2 as (select *, v_predict_udf(img_tensor) as fraud from cte1) select gender, fraud, avg(w_prob), max(w_prob), min(w_prob) from cte2 group by gender, fraud" , labeled_training_set=labeled_training_set).show()

/tmp/ipykernel_226088/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
/tmp/ipykernel_226088/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
/tmp/ipykernel_226088/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
/tmp/ipykernel_226088/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.

+------+-----+-------------------+-----------+-----------+
|gender|fraud|        avg(w_prob)|max(w_prob)|min(w_prob)|
+------+-----+-------------------+-----------+-----------+
| ERROR|  1.0|0.06133333333333333|      0.077|      0.034|
|     F|  0.0|0.06374886877828054|      0.078|      0.031|
|     F|  1.0|           0.063875|      0.078|        0.0|
|     M|  0.0|0.05880672268907564|      0.099|      0.004|
|     M|  1.0|0.06670652173913043|      0.099|        0.0|
+------+-----+-------------------+-----------+-----------+



In [39]:
spark.sql("with cte as (select gender, weight_lbs, predict_udf(img_tensor) as my_predictions from {labeled_training_set}) select gender, my_predictions, max(weight_lbs), min(weight_lbs) as count_predictions from cte group by gender, my_predictions", labeled_training_set=labeled_training_set).show()

+------+--------------+---------------+-----------------+
|gender|my_predictions|max(weight_lbs)|count_predictions|
+------+--------------+---------------+-----------------+
| ERROR|           1.0|            178|              104|
|     F|           0.0|            200|              100|
|     F|           1.0|            226|                0|
|     M|           0.0|            249|              100|
|     M|           1.0|            248|                0|
+------+--------------+---------------+-----------------+



In [45]:
log=[]

def monitor_system_resources(interval=1):
    global monitoring_active
    monitoring_active = True
    print("Monitoring system resources...")
    
    while monitoring_active:
        now = datetime.datetime.now()
        cpu_percent = psutil.cpu_percent(interval=None) # Non-blocking call
        virtual_memory = psutil.virtual_memory()
        memory_percent = virtual_memory.percent
        #print(f"System CPU: {cpu_percent}% | System Memory: {memory_percent}%")
        time.sleep(interval)
        
        
        pid_list = []

        for proc in psutil.process_iter(['pid', 'name', 'cpu_percent', 'memory_percent', 'memory_info']):
            try:
                # Get process information
                pid = proc.info['pid']
                name = proc.info['name']
                cpu_percent_pid = proc.cpu_percent(interval=None) # Calculate CPU usage over a short interval
                memory_percent_pid = proc.info['memory_percent']
                if cpu_percent_pid > 1 or  memory_percent_pid > 1:
                    #memory_percent_pid = proc.info['memory_percent']
                    memory_info = proc.info['memory_info']
                    rss_mb = memory_info.rss / (1024 * 1024) if memory_info else 0 # Resident Set Size in MB
    
                    #print(f"{pid:<8} | {name:<30} | {cpu_percent_pid:<10.2f} | {memory_percent_pid:<10.2f} | {rss_mb:<15.2f}")

                    pid_entry = { "pid": pid,
                                  "name": name,
                                  "cpu_percent": cpu_percent_pid,
                                  "memory_percent": memory_percent_pid,
                                  "rss_mb": rss_mb
                                  }
                    
                    pid_list.append(pid_entry)
                    

            except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
                # Handle cases where a process might disappear or access is denied
                continue

        system_entry = {
            "time" : now.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3] ,
            "cpu_percent" : cpu_percent,
            "memory_percent" : memory_percent,
            #"virtual_memory" : virtual_memory,
            "pids": pid_list
            
        }
        
        log.append(system_entry)
         
    print("Monitoring stopped.")


In [46]:
def my_function_to_trace():
    print("Starting function to trace...")
    spark.sql("with cte as (select gender, weight_lbs, predict_udf(img_tensor) as my_predictions from {labeled_training_set}) select gender, my_predictions, max(weight_lbs), min(weight_lbs) as count_predictions from cte group by gender, my_predictions", labeled_training_set=labeled_training_set).show()
    print("Function to trace finished.")



In [47]:
# Start the monitoring thread
monitor_thread = threading.Thread(target=monitor_system_resources, args=(1,))
monitor_thread.start()

# Run the function to be traced
my_function_to_trace()

# Stop the monitoring thread
global monitoring_active
monitoring_active = False
monitor_thread.join() # Wait for the monitoring thread to finish
print("Program finished.")

Monitoring system resources...
Starting function to trace...


+------+--------------+---------------+-----------------+
|gender|my_predictions|max(weight_lbs)|count_predictions|
+------+--------------+---------------+-----------------+
| ERROR|           1.0|            178|              104|
|     F|           0.0|            200|              100|
|     F|           1.0|            226|                0|
|     M|           0.0|            249|              100|
|     M|           1.0|            248|                0|
+------+--------------+---------------+-----------------+

Function to trace finished.
Monitoring stopped.
Program finished.


In [48]:
pretty_json = json.dumps(log, indent=4)
print(pretty_json)

[
    {
        "time": "2025-09-24 22:40:51.144",
        "cpu_percent": 32.1,
        "memory_percent": 73.4,
        "pids": [
            {
                "pid": 1095,
                "name": "sqlservr",
                "cpu_percent": 0.0,
                "memory_percent": 2.2311405679527736,
                "rss_mb": 1093.53515625
            },
            {
                "pid": 1681,
                "name": "gnome-shell",
                "cpu_percent": 0.0,
                "memory_percent": 1.2477714098079489,
                "rss_mb": 611.5625
            },
            {
                "pid": 2179,
                "name": "firefox",
                "cpu_percent": 0.0,
                "memory_percent": 1.394115167005787,
                "rss_mb": 683.2890625
            },
            {
                "pid": 2380,
                "name": "Privileged Cont",
                "cpu_percent": 0.0,
                "memory_percent": 1.2616151689982682,
                "rss_mb": 61

In [49]:
log

[{'time': '2025-09-24 22:40:51.144',
  'cpu_percent': 32.1,
  'memory_percent': 73.4,
  'pids': [{'pid': 1095,
    'name': 'sqlservr',
    'cpu_percent': 0.0,
    'memory_percent': 2.2311405679527736,
    'rss_mb': 1093.53515625},
   {'pid': 1681,
    'name': 'gnome-shell',
    'cpu_percent': 0.0,
    'memory_percent': 1.2477714098079489,
    'rss_mb': 611.5625},
   {'pid': 2179,
    'name': 'firefox',
    'cpu_percent': 0.0,
    'memory_percent': 1.394115167005787,
    'rss_mb': 683.2890625},
   {'pid': 2380,
    'name': 'Privileged Cont',
    'cpu_percent': 0.0,
    'memory_percent': 1.2616151689982682,
    'rss_mb': 618.34765625},
   {'pid': 5027,
    'name': 'clickhouse-server',
    'cpu_percent': 0.0,
    'memory_percent': 2.102617562366653,
    'rss_mb': 1030.54296875},
   {'pid': 7502,
    'name': 'Isolated Web Co',
    'cpu_percent': 0.0,
    'memory_percent': 2.1581121479983136,
    'rss_mb': 1057.7421875},
   {'pid': 9817,
    'name': 'datagrip',
    'cpu_percent': 0.0,
    '

In [ ]:
import xgboost as xgb

In [ ]:
xgboostmodel = xgb.XGBClassifier()

In [ ]:
def predict_xgboost(tensor_data):
    tensor = torch.Tensor(tensor_data)

    #channel, h, w = tensor.shape

    with torch.no_grad():  # Disable gradient calculation for inference
        prediction = model(tensor.unsqueeze(0).to(device))
        _, y_pred = torch.max(prediction.data, 1)

    return float(y_pred)





predict_udf = udf(predict_tensor, FloatType())